In [65]:
import pandas as pd
import numpy as np
from sklearn import tree
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [66]:
df = pd.read_csv('/content/tabela - tabela.csv.csv', decimal=',')
df = df.drop(columns=['Data'])
df = df.dropna()


In [67]:
X = df.iloc[:,:3]
Y = df.iloc[:,-1]

In [68]:
X

,Temp. Ins. (C),Umi. Ins. (%),Radiacao (KJ/m²)
0,24.4,83,0.0
1,24.5,85,0.0
2,24.3,86,0.0
3,24.2,84,0.0
4,23.9,86,0.0
...,...,...,...
86,26.5,86,2076.3
87,25.8,86,1464.3
88,26.6,85,911.2
89,27.5,78,1327.1


In [69]:
Y

,Temperaturas alvos
0,23.0
1,23.1
2,23.2
3,23.3
4,23.4
...,...
86,31.6
87,31.7
88,31.8
89,31.9


In [70]:
corte = int(len(X) * 0.8)

X_treino = X.iloc[:corte]
X_teste = X.iloc[corte:]

Y_treino = Y.iloc[:corte]
Y_teste = Y.iloc[corte:]

In [71]:
arvore_teste = tree.DecisionTreeRegressor(random_state=42)

arvore_teste.fit(X_treino, Y_treino)

DecisionTreeRegressor(random_state=42)

In [72]:
entradas = X_teste.iloc[:20]

previsoes = arvore_teste.predict(entradas)

In [73]:
comparacao = pd.DataFrame({
    'Temperatura real': Y_teste.iloc[:20].values,
    'Temperatura prevista': previsoes,
    'Erro absoluto': np.abs(
        Y_teste.iloc[:20].values - previsoes
    )
})

comparacao

,Temperatura real,Temperatura prevista,Erro absoluto
0,30.2,30.1,0.1
1,30.3,30.0,0.3
2,30.4,30.0,0.4
3,30.5,30.0,0.5
4,30.6,30.0,0.6
5,30.7,30.0,0.7
6,30.8,30.0,0.8
7,30.9,30.0,0.9
8,31.0,30.0,1.0
9,31.1,29.9,1.2


In [74]:
resultados = []

for profundidade in [None, 2, 3, 4, 5,6,7,8, 9, 10]:

    modelo = tree.DecisionTreeRegressor(
        max_depth=profundidade,
        random_state=42
    )

    modelo.fit(X_treino, Y_treino)

    predicoes = modelo.predict(X_teste)

    erro_absoluto_medio = mean_absolute_error(Y_teste, predicoes)

    raiz_do_erro_quadratico_medio = np.sqrt(mean_squared_error(Y_teste, predicoes))

    resultados.append({
        'Profundidade configurada': profundidade,
        'Profundidade real': modelo.get_depth(),
        'Nós': modelo.tree_.node_count,
        'Folhas': modelo.get_n_leaves(),
        'erro absoluto medio': erro_absoluto_medio,
        'Raiz do erro quadratico medio': raiz_do_erro_quadratico_medio
    })
pd.DataFrame(resultados)

,Profundidade configurada,Profundidade real,Nós,Folhas,erro absoluto medio,Raiz do erro quadratico medio
0,NaN,11,143,72,1.963158,2.590367
1,2.0,2,7,4,2.215851,2.922744
2,3.0,3,15,8,1.961340,2.678324
3,4.0,4,31,16,2.033094,2.783283
4,5.0,5,55,28,2.006579,2.727691
5,6.0,6,83,42,2.041729,2.792323
6,7.0,7,105,53,2.023684,2.706158
7,8.0,8,121,61,1.955263,2.572246
8,9.0,9,133,67,1.942105,2.566484
9,10.0,10,141,71,1.915789,2.524407


In [76]:
for profundidade in [3,8,9,10, None]:

    modelo = tree.DecisionTreeRegressor(
        max_depth=profundidade,
        random_state=42
    )

    modelo.fit(X_treino, Y_treino)

    treino = modelo.predict(X_treino)
    teste = modelo.predict(X_teste)

    print("\nProfundidade:", profundidade)
    print("Nós:", modelo.tree_.node_count)

    print("MAE treino:",
          mean_absolute_error(Y_treino, treino))

    print("MAE teste:",
          mean_absolute_error(Y_teste, teste))

    print("RMSE treino:",
          np.sqrt(mean_squared_error(Y_treino, treino)))

    print("RMSE teste:",
          np.sqrt(mean_squared_error(Y_teste, teste)))


Profundidade: 3
Nós: 15
MAE treino: 0.7867760942760943
MAE teste: 1.9613397129186596
RMSE treino: 0.9982459363778795
RMSE teste: 2.678323552316203

Profundidade: 8
Nós: 121
MAE treino: 0.1263888888888889
MAE teste: 1.9552631578947368
RMSE treino: 0.36052302541148706
RMSE teste: 2.5722455886678186

Profundidade: 9
Nós: 133
MAE treino: 0.08101851851851845
MAE teste: 1.9421052631578946
RMSE treino: 0.2977150638769547
RMSE teste: 2.566484386327561

Profundidade: 10
Nós: 141
MAE treino: 0.031944444444444456
MAE teste: 1.91578947368421
RMSE treino: 0.1916666666666667
RMSE teste: 2.5244071737632514

Profundidade: None
Nós: 143
MAE treino: 0.0
MAE teste: 1.963157894736842
RMSE treino: 0.0
RMSE teste: 2.5903667693977237


Olha, o modelo com profundidade 10 teve melhor desempenho, profundidade 11 OVERFITTING! modelo com profundidade 3 é mais simples, foi bem no desempenho é bem mais tranquilo para implementar C ficou menos nós e foi bem. Para implementar no atmega328p profundidade 3 com 15 nós é bem melhor. Discutir com equipe na reunião!

In [78]:
modelo_3 = tree.DecisionTreeRegressor(
    max_depth=3,
    random_state=42
)

modelo_10 = tree.DecisionTreeRegressor(
    max_depth=10,
    random_state=42
)

In [79]:
modelo_3.fit(X_treino, Y_treino)

modelo_10.fit(X_treino, Y_treino)

DecisionTreeRegressor(max_depth=10, random_state=42)

In [80]:
def exportar_estrutura(modelo, nome_arquivo):

    estrutura = modelo.tree_
    nos = []

    for no in range(estrutura.node_count):

        esquerda = estrutura.children_left[no]
        direita = estrutura.children_right[no]

        folha = esquerda == -1

        indice = estrutura.feature[no]

        nos.append({
            'no': no,
            'tipo': 'folha' if folha else 'decisao',
            'variavel': X.columns[indice] if not folha else '',
            'feature_index': int(indice) if not folha else None,
            'limite': float(estrutura.threshold[no]) if not folha else None,
            'filho_esquerdo': int(esquerda),
            'filho_direito': int(direita),
            'previsao': float(estrutura.value[no][0][0]) if folha else None,
            'amostras': int(estrutura.n_node_samples[no])
        })

    df_nos = pd.DataFrame(nos)

    df_nos.to_csv(nome_arquivo, index=False)

    print("Arquivo gerado:", nome_arquivo)
    print("Total de nós:", estrutura.node_count)
    print("Folhas:", modelo.get_n_leaves())

In [81]:
exportar_estrutura(modelo_3, 'estrutura_arvore_3.csv')

exportar_estrutura(modelo_10, 'estrutura_arvore_10.csv')

Arquivo gerado: estrutura_arvore_3.csv
Total de nós: 15
Folhas: 8
Arquivo gerado: estrutura_arvore_10.csv
Total de nós: 141
Folhas: 71


In [82]:
for nome, modelo in [('Profundidade 3', modelo_3),
                     ('Profundidade 10', modelo_10)]:

    predicoes = modelo.predict(X_teste)

    print("\n", nome)
    print("Profundidade:", modelo.get_depth())
    print("Nós:", modelo.tree_.node_count)
    print("Folhas:", modelo.get_n_leaves())
    print("MAE:", mean_absolute_error(Y_teste, predicoes))
    print("RMSE:", np.sqrt(mean_squared_error(Y_teste, predicoes)))


 Profundidade 3
Profundidade: 3
Nós: 15
Folhas: 8
MAE: 1.9613397129186596
RMSE: 2.678323552316203

 Profundidade 10
Profundidade: 10
Nós: 141
Folhas: 71
MAE: 1.91578947368421
RMSE: 2.5244071737632514
